# Fine-tune bge-reranker-v2-m3 with traindataset to get top 10 results
Traindataset was mined hard negatives with best Vietnamese Biencoder in vnese_biencoder_ft_no_val.ipynb

In [ ]:
!pip install -U sentence-transformers transformers accelerate py_vncorenlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.8 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.2.0
    Uninstalling sentence-transformers-5.2.0:
      Successfully uninstalled sentence-transformers-5.2.0


In [3]:
import json
import numpy as np
import pandas as pd
import py_vncorenlp

In [4]:
py_vncorenlp.download_model(save_dir='/kaggle/working')

--2026-03-07 16:28:58--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 4.27M 6s
    50K .......... .......... .......... .......... ..........  0% 5.00M 6s
   100K .......... .......... .......... .......... ..........  0% 14.5M 4s
   150K .......... .......... .......... .......... ..........  0%  216M 3s
   200K .......... .......... .......... .......... ..........  0% 20.3M 3s
   250K .......... .......... .......... .......... ..........  1% 9.14M 3s
   300K .......... .......... .......... .......... ..........  1% 19.4M 3s
   350K ..

In [5]:
# Load the word and sentence segmentation component
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir='/kaggle/working')

text = "Ông Nguyễn Khắc Chúc  đang làm việc tại Đại học Quốc gia Hà Nội. Bà Lan, vợ ông Chúc, cũng làm việc tại đây."

output = rdrsegmenter.word_segment(text)

print(output)
# ['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', 'Bà Lan , vợ ông Chúc , cũng làm_việc tại đây .']

2026-03-07 16:29:01 INFO  WordSegmenter:24 - Loading Word Segmentation model
['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', 'Bà Lan , vợ ông Chúc , cũng làm_việc tại đây .']


In [6]:
# segment laws in corpus
corpus_Id2Text = {}
corpus_Text2Id = {}


with open("/kaggle/input/datasets/duongquanganh/chunked-corpus/chunked_corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

for raw in corpus:
    corpus_Id2Text[raw['chunk_id']] = raw['content_Article']
    corpus_Text2Id[raw['content_Article']] = raw['chunk_id']

In [7]:
corpus_text = []

for raw in corpus:
    corpus_text.append(raw['content_Article'])

In [8]:
with open("/kaggle/input/datasets/duongquanganh/traindata-retrieval/train.json", "r", encoding = "utf-8") as f:
    train_data = json.load(f)

# segment question in train data
for q in train_data:
    output = rdrsegmenter.word_segment(q['question'])
    q['question'] = " ".join(output)

In [9]:
train = {'query': [], 'answer': []}
for q in train_data:
    for rel_law in q['relevant_laws']:
        pattern_rel_law = str(rel_law) + '_'
        i = 0
        while True:
            chunked_rel_law = pattern_rel_law + str(i)
            if chunked_rel_law not in corpus_Id2Text:
                break
            train['query'].append(q['question'])
            train['answer'].append(corpus_Id2Text[chunked_rel_law])
            i+=1

In [10]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("/kaggle/input/datasets/duongquanganh/checkpoint400-vnese-biencoder/checkpoint-400", device = 'cuda')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.9 MB/s eta 0:00:00


In [ ]:
from sentence_transformers.util import mine_hard_negatives
from datasets import Dataset

# Load a dataset to mine hard negatives from
train_dataset = Dataset.from_dict(train)
num_hard_negatives = 5

train_dataset = mine_hard_negatives(
    dataset=train_dataset,
    model=model,
    relative_margin=0.05,
    num_negatives=num_hard_negatives,
    sampling_strategy="top",
    batch_size=128,
    use_faiss=True,
    output_format = 'labeled-pair'
)

Setting range_max to 157 based on the provided parameters.
Found 2189 unique queries out of 8933 total queries.
Found an average of 4.081 positives per query.


Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00,  3.86it/s]


Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count           8,907         44,310               
Mean           0.7114         0.5485         0.1631
Median         0.7187         0.5409         0.1460
Std            0.0861         0.0951         0.0853
Min            0.1268         0.2170         0.0115
25%            0.6554         0.4818         0.0933
50%            0.7187         0.5409         0.1460
75%            0.7747         0.6143         0.2215
Max            0.9368         0.8309         0.5337
Skipped 6,341 potential negatives (1.83%) due to the relative_margin of 0.05.
Could not find enough negatives for 51 samples (0.26%). Consider adjusting the range_max and relative_margin parameters if you'd like to find more valid negatives.


In [13]:
train_dataset[0]

{'query': 'Thưa luật_sư tôi có đăng_ký kết_hôn trên pháp_luật nhưng nay vợ_chồng bỏ nhau theo phong_tục tập_quán như_vậy tôi có được phép kết_hôn với người khác không ạ ?',
 'answer': '1 . Quan_hệ hôn_nhân và gia_đình được xác_lập , thực_hiện theo quy_định của Luật này được tôn_trọng và được pháp_luật bảo_vệ . 2 . Cấm các hành_vi sau đây : a ) Kết_hôn giả_tạo , ly_hôn giả_tạo ; b ) Tảo_hôn , cưỡng_ép kết_hôn , lừa_dối kết_hôn , cản_trở kết_hôn ; c ) Người đang có vợ , có chồng mà kết_hôn hoặc chung sống như vợ_chồng với người khác hoặc chưa có vợ , chưa có chồng mà kết_hôn hoặc chung sống như vợ_chồng với người đang có chồng , có vợ ; d ) Kết_hôn hoặc chung sống như vợ_chồng giữa những người cùng dòng máu về trực_hệ ; giữa những người có họ trong phạm_vi ba đời ; giữa cha , mẹ nuôi với con_nuôi ; giữa người đã từng là cha , mẹ nuôi với con_nuôi , cha chồng với con dâu , mẹ_vợ với con rể , cha dượng với con_riêng của vợ , mẹ_kế với con_riêng của chồng ; đ ) Yêu_sách của_cải trong kết_hô

In [14]:
import torch
torch.cuda.empty_cache()

In [ ]:
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer
from sentence_transformers.cross_encoder.losses.BinaryCrossEntropyLoss import BinaryCrossEntropyLoss

model = CrossEncoder("BAAI/bge-reranker-v2-m3")
loss = BinaryCrossEntropyLoss(model, pos_weight = torch.tensor(num_hard_negatives))
train_batch_size = 6
num_epochs = 4

args = CrossEncoderTrainingArguments(
    output_dir="bge-reranker-finetuned",
    save_steps=200,
    learning_rate=2e-5,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=train_batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True
)

args = CrossEncoderTrainingArguments(
    output_dir="bge-reranker-finetuned",
    num_train_epochs=num_epochs,
    per_device_train_batch_size=train_batch_size,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    bf16=False,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    logging_steps=50,
    logging_first_step=True,
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss
)

trainer.train()

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13220, training_loss=1.079819584145308, metrics={'train_runtime': 24822.1687, 'train_samples_per_second': 3.195, 'train_steps_per_second': 0.533, 'total_flos': 0.0, 'train_loss': 1.079819584145308, 'epoch': 4.0})

In [16]:
with open("/kaggle/input/datasets/duongquanganh/privatetest/DRILL_PrivateTest/private_test.json","r",encoding = "utf-8") as f:
    file = json.load(f)

qid_to_text = {}
text_to_qid = {}
for q in file:
    output = rdrsegmenter.word_segment(q['question'])
    q['question'] = " ".join(output)
    qid_to_text[q['qid']] = q['question']
    text_to_qid[q['question']] = q['qid']

In [17]:
with open("/kaggle/input/datasets/hienhoangthu/top100-input-rerank-r0-9336/top100_results_input_rerank_r0.9336.json", "r", encoding="utf-8") as f:
    ce_input_data = json.load(f)

In [ ]:
reranked_results = []

print(f"Re-ranking {len(ce_input_data)} questions...")

for i, item in enumerate(ce_input_data):
    qid = item['qid']
    question = qid_to_text[qid]
    candidates = item['relevant_laws']
    
    pairs = [[question, corpus_Id2Text[candidate]] for candidate in candidates]

    scores = model.predict(pairs)
    
    if isinstance(scores, float):
        scores = [scores]
    
    candidate_score_pairs = [(candidates[j], scores[j], j) for j in range(len(candidates))]

    candidate_score_pairs.sort(key=lambda x: x[1], reverse=True)
    
    reranked_results.append({
        'qid': qid,
        'question': question,
        'reranked_candidates': [
            {
                'chunk_id': chunk_id,
                'score': score,
                'original_rank': orig_idx
            }
            for chunk_id, score, orig_idx in candidate_score_pairs
        ],
        'num_candidates': len(candidates)
    })
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(ce_input_data)} questions")

print(f"\nRe-ranking complete! Processed {len(reranked_results)} questions")

Re-ranking 627 questions...
Processed 10/627 questions
Processed 20/627 questions
Processed 30/627 questions
Processed 40/627 questions
Processed 50/627 questions
Processed 60/627 questions
Processed 70/627 questions
Processed 80/627 questions
Processed 90/627 questions
Processed 100/627 questions
Processed 110/627 questions
Processed 120/627 questions
Processed 130/627 questions
Processed 140/627 questions
Processed 150/627 questions
Processed 160/627 questions
Processed 170/627 questions
Processed 180/627 questions
Processed 190/627 questions
Processed 200/627 questions
Processed 210/627 questions
Processed 220/627 questions
Processed 230/627 questions
Processed 240/627 questions
Processed 250/627 questions
Processed 260/627 questions
Processed 270/627 questions
Processed 280/627 questions
Processed 290/627 questions
Processed 300/627 questions
Processed 310/627 questions
Processed 320/627 questions
Processed 330/627 questions
Processed 340/627 questions
Processed 350/627 questions
P

In [19]:
reranked_results[0]

{'qid': 1497,
 'question': 'Phạm_nhân không biết chữ có được tạo điều_kiện học văn_hoá nhằm xoá mù_chữ hay không ?',
 'reranked_candidates': [{'chunk_id': '57161_0',
   'score': np.float32(0.9997595),
   'original_rank': 87},
  {'chunk_id': '13860_0', 'score': np.float32(0.9997464), 'original_rank': 26},
  {'chunk_id': '6917_0', 'score': np.float32(0.9995777), 'original_rank': 36},
  {'chunk_id': '15220_0', 'score': np.float32(0.9993735), 'original_rank': 56},
  {'chunk_id': '44689_0', 'score': np.float32(0.99846566), 'original_rank': 8},
  {'chunk_id': '45604_0', 'score': np.float32(0.9983834), 'original_rank': 18},
  {'chunk_id': '45605_0', 'score': np.float32(0.9922969), 'original_rank': 65},
  {'chunk_id': '13860_1', 'score': np.float32(0.9847002), 'original_rank': 97},
  {'chunk_id': '44678_1', 'score': np.float32(0.9443234), 'original_rank': 67},
  {'chunk_id': '43898_0',
   'score': np.float32(0.16192591),
   'original_rank': 33},
  {'chunk_id': '34472_3',
   'score': np.float32

In [ ]:
top_10_results = []

for result in reranked_results:
    top_10_results.append({
        'qid': result['qid'],
        'relevant_laws': list(set(int(t['chunk_id'].split("_")[0]) for t in result['reranked_candidates'][:10]))
    })

with open("reranked_top10_results_submit.json", "w", encoding="utf-8") as f:
    json.dump(top_10_results, f, ensure_ascii=False, indent=2)

print(f"Saved top 10 reranked results for {len(top_10_results)} questions to 'reranked_top10_results_submit.json'")

Saved top 10 reranked results for 627 questions to 'reranked_top10_results_submit.json'


In [ ]:
top_10_results = []

for result in reranked_results:
    top_10_results.append({
        'qid': result['qid'],
        'relevant_laws': list(set(t['chunk_id'] for t in result['reranked_candidates'][:10]))
    })

with open("reranked_top10_results_input_judge.json", "w", encoding="utf-8") as f:
    json.dump(top_10_results, f, ensure_ascii=False, indent=2)

print(f"Saved top 10 reranked results for {len(top_10_results)} questions to 'reranked_top10_results_input_judge.json'")

Saved top 10 reranked results for 627 questions to 'reranked_top10_results_input_judge.json'
